[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/l07_rnn_companion.ipynb)

# L07 Companion: Recurrent Networks and LSTMs

This notebook accompanies the lecture **Recurrent Networks and LSTMs**. The slides are conceptual; here you run the computations.
The modules follow the lecture's dependency chain:

| Module | What you do | Lecture Section & Slides |
|---|---|---|
| N1 | Build shifted targets (BOS/EOS) and compute next-token loss by hand | **Predicting Language** → *The Next-Token Objective* |
| N2 | Run the scalar counter, a vector tanh recurrence, and match `nn.RNN` | **Recurrence** → *Building a Prefix State* |
| N3 | Train a tiny character LM; perplexity; greedy vs. sampled generation | **Using Recurrent Models** → *Training & Generation*, *Evaluation* |
| N4 | Inspect BPTT with autograd; gradient norm vs. distance; clipping; TBPTT | **Learning Through Time** → *BPTT*, *Long-Distance Gradients* |
| N5 | Compute one LSTM step from the six equations; control the forget gate | **Long Short-Term Memory** → *Why Carry Path Needs Gates*, *Cell State* |
| N6 | Map the equations to `nn.LSTM` shapes and state representations | **Long Short-Term Memory** → *Six Equations*, **Encoder-Decoder Models** |
| N7 | Deep Dive: Replaying Karpathy's *War and Peace* LSTM training | **Using Recurrent Models** → *Learning to Generate Text* |

**Notation** (as on the slides): row vectors, $\mathbf{h}_t = \tanh(\mathbf{x}_t W_{xh} + \mathbf{h}_{t-1} W_{hh} + \mathbf{b}_h)$.
Step $t$ reads $x_t$ and predicts $x_{t+1}$; $x_1=$ BOS is input only. Loss = mean negative log-likelihood (natural log), $\mathrm{PPL}=\exp(\mathcal{L})$.

Modules N1–N6 run in under a minute on CPU with no downloads. Module N7 optionally streams Karpathy's *War and Peace* corpus and leverages MPS/CUDA if available.

In [ ]:
# Dependencies: torch, matplotlib; ipywidgets is optional (sliders in N4/N5).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, IntSlider, FloatSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

torch.manual_seed(0)
print("torch", torch.__version__, "| widgets:", HAVE_WIDGETS)

## N1 — Self-Supervised Targets *(Predicting Language: The Next-Token Objective)*

A causal LM needs no labels: the targets are the input sequence **shifted by one**.
For the word *hello* (from slide *"Text Provides Its Own Learning Targets"*), the lecture sets up:

| input  | BOS | h | e | l | l | o |
|---|---|---|---|---|---|---|
| target | h | e | l | l | o | EOS |

In [ ]:
BOS, EOS = "<s>", "</s>"
vocab = [BOS, EOS, "e", "h", "l", "o"]
stoi = {s: i for i, s in enumerate(vocab)}

seq = [BOS] + list("hello") + [EOS]
ids = torch.tensor([stoi[s] for s in seq])
inputs, targets = ids[:-1], ids[1:]       # the shift
for x, y in zip(inputs, targets):
    print(f"{vocab[x]:>4} -> {vocab[y]}")
assert len(inputs) == len(targets) == 6   # M = 6 predicted positions

Suppose some model produced these scores (logits) $\mathbf{o}_t\in\mathbb{R}^{V}$ for each of the $M=6$ positions.
The loss is the mean over positions of $-\log p_\theta(x_{t+1}\mid x_{1:t})$, where $p$ is the softmax of the scores.
Compute it by hand and compare with `F.cross_entropy`.

In [ ]:
torch.manual_seed(1)
V, M = len(vocab), len(inputs)
logits = torch.randn(M, V)                # (M, V): one score row per position

log_probs = logits - logits.logsumexp(dim=-1, keepdim=True)   # log softmax
nll_manual = -log_probs[torch.arange(M), targets].mean()
nll_torch = F.cross_entropy(logits, targets)                   # same convention: mean
print(f"manual {nll_manual:.4f}  torch {nll_torch:.4f}  PPL {nll_manual.exp():.2f}")
assert torch.allclose(nll_manual, nll_torch)

**Check yourself.** A model that assigns uniform probability $1/V$ everywhere has loss $\log V$ and perplexity $V$. Verify this with `logits = torch.zeros(M, V)`.
Why is BOS never a target, and why is EOS never an input?

<details>
<summary><b>Click to expand: Solution & explanation</b></summary>

- **Uniform baseline:** When logits are all zero, the softmax probabilities are $p_k = 1/V$ for all tokens $k$. The negative log-likelihood at every position is $-\log(1/V) = \log V$. Perplexity is $\exp(\mathcal{L}) = \exp(\log V) = V$ (the effective number of choices is the full vocabulary size).
- **BOS boundary condition:** Sequence generation always starts with BOS as given context ($p(x_1 = \text{BOS}) = 1$). A model is never asked to predict BOS, because BOS announces the start of generation; predicting it would waste probability mass on a non-generable token.
- **EOS boundary condition:** EOS signals that generation is complete. Once EOS is emitted, generation terminates; feeding EOS as an input would ask the model to predict what follows the end of the text, which is undefined in causal sequence generation.
</details>

## N2 — Manual Recurrence *(Recurrence: Building a Prefix State)*

### 2.1 The scalar counter *(Slide: "A Scalar Recurrent State")*
$h_t = w\cdot x_t + r\cdot h_{t-1}$, with $w=r=1$, identity activation, $h_0=0$.

In [ ]:
def counter(xs, w=1.0, r=1.0, h=0.0):
    states = []
    for x in xs:
        h = w * x + r * h
        states.append(h)
    return states

print(counter([1, 0, 1, 1]))      # slide B01: 1, 1, 2, 3

Try $r=0.5$ (the state forgets) and $r=1.5$ (the state explodes). This is the scalar version of the gradient story in N4.

### 2.2 A vector tanh recurrence, then `nn.RNN` *(Slides: "Vanilla RNN Update", "Unrolling Shares Parameters Across Time")*
The **same** parameters $W_{xh}\in\mathbb{R}^{d\times h}$, $W_{hh}\in\mathbb{R}^{h\times h}$, $\mathbf{b}_h\in\mathbb{R}^{h}$ are used at every step.

In [ ]:
torch.manual_seed(0)
d, hdim, T = 3, 4, 5
X = torch.randn(T, d)                     # one sequence: rows x_1 ... x_T
W_xh = torch.randn(d, hdim) * 0.5
W_hh = torch.randn(hdim, hdim) * 0.5
b_h = torch.zeros(hdim)

def rnn_star(X, h=None):
    # RNN*: returns all states h_1..h_T (Goldberg's interface); RNN would return only the last
    h = torch.zeros(W_hh.shape[0]) if h is None else h
    states = []
    for x_t in X:
        h = torch.tanh(x_t @ W_xh + h @ W_hh + b_h)
        states.append(h)
    return torch.stack(states)            # (T, h)

H = rnn_star(X)
print("states", tuple(H.shape), "| final state h_T =", H[-1])

PyTorch stores weights as `(out_features, in_features)` and computes `x @ W.T`. To make `nn.RNN` compute **exactly** our recurrence, copy the transposed matrices (and set one of its two bias vectors to zero).

In [ ]:
rnn = nn.RNN(input_size=d, hidden_size=hdim, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(W_xh.T)        # (h, d)
    rnn.weight_hh_l0.copy_(W_hh.T)        # (h, h)
    rnn.bias_ih_l0.copy_(b_h)
    rnn.bias_hh_l0.zero_()

out, h_n = rnn(X.unsqueeze(0))            # batch of 1: (1, T, d)
print("output", tuple(out.shape), "h_n", tuple(h_n.shape))
assert torch.allclose(out[0], H, atol=1e-6)      # output = RNN*: all states
assert torch.allclose(h_n[0, 0], H[-1], atol=1e-6)  # h_n = RNN: final state
print("nn.RNN matches the manual loop.")

**Check yourself.** Count the parameters: $dh + h^2 + h$. Does the number depend on $T$? Why does that matter for sequences of arbitrary length?

<details>
<summary><b>Click to expand: Solution & explanation</b></summary>

- **Breakdown:**
  - $W_{xh} \in \mathbb{R}^{d \times h}$: maps input token embedding to hidden state $\to dh$ weights.
  - $W_{hh} \in \mathbb{R}^{h \times h}$: maps previous hidden state to next hidden state $\to h^2$ weights.
  - $\mathbf{b}_h \in \mathbb{R}^{h}$: bias vector $\to h$ weights.
- **Sequence invariance:** Notice that $T$ (the sequence length) does not appear in $dh + h^2 + h$. Because the **same** transition matrices are reused at every time step $t=1, \dots, T$, the model can process sequences of any length without changing its parameter footprint. By contrast, a feed-forward network spanning a sequence of length $T$ would need an input layer of size $T \cdot d$, tying the parameter count directly to a fixed maximum length.
</details>

## N3 — A Tiny Character LM *(Using Recurrent Models: Training and Evaluation)*

A recurrent LM = embedding → RNN → a **separate** shared head that turns each state into scores over the vocabulary.
We train it with teacher forcing on a short built-in text. Every training window starts with BOS, as at generation time.

In [ ]:
text = (
    "the cat sat on the mat. the dog sat on the log. the cat saw the dog. "
    "a dog and a cat sat on a log. the dog ran to the cat. the cat ran to the mat. "
    "a cat saw a rat on the mat. the rat ran to the log. the dog saw the rat. "
    "the cat and the dog sat on the mat. a rat sat on a log and saw the cat. "
    "the dog ran and the cat sat. the rat saw a dog on the log and ran. "
) * 4
chars = sorted(set(text))
V = len(chars) + 1                         # + BOS
BOS_ID = len(chars)
cstoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([cstoi[c] for c in text])
print("vocabulary size", V, "| corpus length", len(data))

class CharLM(nn.Module):
    def __init__(self, V, d=16, hdim=64):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.rnn = nn.RNN(d, hdim, batch_first=True)
        self.head = nn.Linear(hdim, V)     # separate head: state -> scores
    def forward(self, x, h=None):
        H, h = self.rnn(self.emb(x), h)    # H: all states (B, T, h)
        return self.head(H), h             # scores (B, T, V)

In [ ]:
def batches(data, T=32, B=16):
    starts = torch.randint(0, len(data) - T - 1, (B,))
    bos = torch.tensor([BOS_ID])
    x = torch.stack([torch.cat([bos, data[s:s + T - 1]]) for s in starts])   # BOS + window
    y = torch.stack([data[s:s + T] for s in starts])                         # shifted targets
    return x, y

torch.manual_seed(0)
lm = CharLM(V)
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
history = []
for step in range(600):
    x, y = batches(data)
    scores, _ = lm(x)
    loss = F.cross_entropy(scores.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(lm.parameters(), 1.0)          # see D06 / N4
    opt.step()
    history.append(loss.item())
    if step % 150 == 0 or step == 599:
        print(f"step {step:3d}  loss {loss.item():.3f}  PPL {math.exp(loss.item()):6.2f}")

plt.plot(history); plt.xlabel("step"); plt.ylabel("mean NLL (nats)"); plt.title("Training loss"); plt.show()

Perplexity starts near $V$ (uniform guessing) and drops far below. Perplexities are comparable only for the **same data and tokenization**.

**Generation** is autoregressive: the model's own output becomes the next input (as on slide *"Training and Generation"*). Greedy decoding picks the argmax; sampling draws from $p_\theta$ (with a temperature).

In [ ]:
@torch.no_grad()
def generate(prefix, n=60, temperature=None):
    x = torch.tensor([[BOS_ID] + [cstoi[c] for c in prefix]])
    scores, h = lm(x)
    out = prefix
    for _ in range(n):
        last = scores[0, -1].clone()
        last[BOS_ID] = -float("inf")                         # BOS is input only, never generated
        if temperature is None:
            nxt = last.argmax().item()                       # greedy
        else:
            nxt = torch.multinomial(F.softmax(last / temperature, -1), 1).item()
        out += chars[nxt]
        scores, h = lm(torch.tensor([[nxt]]), h)            # feed own output back
    return out

torch.manual_seed(0)
print("greedy      :", generate("the "))
print("sampled T=1 :", generate("the ", temperature=1.0))
print("sampled T=2 :", generate("the ", temperature=2.0))

**Check yourself.** Why does greedy decoding tend to loop? What changes at temperature 0.5 vs. 2? Like Karpathy's samples on the slide *"A Character Model Plays the Shannon Game"*, these outputs show *local regularities* of the corpus, not language understanding.

<details>
<summary><b>Click to expand: Solution & explanation</b></summary>

- **Greedy loops:** Greedy decoding deterministically selects $\arg\max_k p_\theta(x_{t+1}=k \mid x_{1:t})$. Because recurrent states form a continuous deterministic dynamic system, if the hidden state revisits a similar region after generating a frequent phrase (e.g. `"the cat sat on the mat and the cat sat..."`), the greedy argmax will reproduce the exact same token sequence, creating an unbreakable periodic loop.
- **Temperature scaling:** Softmax with temperature divides logits by $\tau$: $p_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$.
  - At **low temperature ($\tau = 0.5$)**, high-probability tokens are amplified and low-probability tokens are suppressed. Text becomes safer, more grammatical, but repetitive.
  - At **high temperature ($\tau = 2.0$)**, the distribution flattens toward uniform randomness. Output becomes diverse and creative, but quickly devolves into incoherent spelling errors and noise.
</details>

## N4 — BPTT and the Gradient Problem *(Learning Through Time: BPTT & Long-Distance Gradients)*

### 4.1 Autograd follows the unrolled graph
We keep every state with `.retain_grad()` and compute a loss **only at the last step**, so each $\partial\mathcal{L}_T/\partial\mathbf{h}_s$ measures one long-distance path (see the plan's caution: a sum of all step losses would add short local paths and mask the effect).

In [ ]:
def state_grads(T=20, r=1.0, linear=False, hdim=8, seed=0):
    # vanilla RNN (tanh, or identity if linear) with recurrent matrix scaled to spectral norm r
    g = torch.Generator().manual_seed(seed)
    W_hh = torch.linalg.qr(torch.randn(hdim, hdim, generator=g))[0] * r   # orthogonal * r
    W_xh = torch.randn(hdim, hdim, generator=g) * 0.5
    X = torch.randn(T, hdim, generator=g)
    h = torch.zeros(hdim, requires_grad=True)
    states = []
    for x_t in X:
        a = x_t @ W_xh + h @ W_hh
        h = a if linear else torch.tanh(a)
        h.retain_grad()
        states.append(h)
    loss_T = states[-1].sum()             # a loss at the final step only
    loss_T.backward()
    # distance T - s from the loss; norm of dL_T / dh_s
    return [(T - 1 - s, states[s].grad.norm().item()) for s in range(T)]

for dist, n in state_grads(T=6)[::-1]:
    print(f"distance {dist}: ||dL_T/dh_s|| = {n:.4f}")

### 4.2 Interactive: sequence length and recurrence strength
The recurrent matrix is orthogonal times $r$. With $r<1$ (and tanh derivatives $\le 1$) the norms **vanish** with distance. Try a large $r$: the tanh saturates ($1-h^2\approx 0$) and still cuts the gradient. Tick `linear` (identity activation, like the counter) to see $r>1$ **explode** and $r<1$ vanish geometrically.

In [ ]:
def plot_grads(T=30, r=1.0, linear=False):
    dist, norms = zip(*state_grads(T=T, r=r, linear=linear))
    plt.figure(figsize=(6, 3.2))
    plt.semilogy(dist, norms, "o-")
    plt.xlabel("distance from the loss (T - s)"); plt.ylabel(r"$\|\partial L_T/\partial h_s\|$")
    plt.title(f"T={T}, r={r}, {'linear' if linear else 'tanh'}"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

if HAVE_WIDGETS:
    interact(plot_grads, T=IntSlider(30, 5, 80, 5), r=FloatSlider(value=1.0, min=0.3, max=3.0, step=0.1), linear=False)
else:
    for r, linear in ((0.5, False), (1.0, False), (3.0, False), (1.2, True)):
        plot_grads(T=30, r=r, linear=linear)

**Check yourself.** Why does the plain RNN at $r=1$ still lose gradient with distance? (Hint: see slide *"The Chain Rule Across Time Multiplies Jacobians"*, the factor $r(1-h_t^2)$.)

<details>
<summary><b>Click to expand: Solution & explanation</b></summary>

Even when the recurrent weight matrix $W_{hh}$ has spectral norm $r = \|W_{hh}\|_2 = 1$, the Jacobian of the recurrent step is:
$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \operatorname{diag}\bigl(1 - \mathbf{h}_t^2\bigr) W_{hh}^\top$$
The derivative of $\tanh(a)$ is $1 - \tanh^2(a) = 1 - h_t^2 \in (0, 1]$. Unless the hidden state is exactly $\mathbf{0}$ (where $\tanh'(0) = 1$), every step contracts the gradient by a factor strictly less than 1. Across 30 steps, multiplying 30 numbers that are each, say, $\le 0.8$ produces $0.8^{30} \approx 0.0012$, causing the gradient signal to vanish completely.
</details>

### 4.3 Clipping bounds the step, not the vanishing *(Slide: "Gradient Clipping Bounds Exploding Steps")*

In [ ]:
p = nn.Parameter(torch.zeros(3))
p.grad = torch.tensor([300.0, -400.0, 0.0])          # an exploding gradient, norm 500
before = p.grad.norm().item()
nn.utils.clip_grad_norm_([p], max_norm=1.0)
print(f"norm before {before:.1f}, after {p.grad.norm():.3f}; direction kept: {p.grad}")

Clipping rescales a huge gradient to norm 1 and keeps its direction. It cannot **create** a gradient that has vanished.

### 4.4 Truncated BPTT with `detach` *(Slide: "Truncated BPTT Backpropagates Through Chunks")*
Process a long sequence in chunks: the state flows **forward** across chunk boundaries, but `detach()` cuts the graph, so gradients do not cross them.

In [ ]:
torch.manual_seed(0)
long = data[:1 + 4 * 32].unsqueeze(0)                 # 4 chunks of length 32
h = None
for k in range(4):
    x = long[:, k * 32:(k + 1) * 32]
    y = long[:, k * 32 + 1:(k + 1) * 32 + 1]
    scores, h = lm(x, h)
    loss = F.cross_entropy(scores.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    h = h.detach()                                    # keep value, cut gradient path
    print(f"chunk {k}: loss {loss.item():.3f}, state requires grad: {h.requires_grad}")

Remove `h = h.detach()` and run again: the second `backward()` fails, because it would have to traverse the previous chunk's already-freed graph.

<details>
<summary><b>Click to expand: PyTorch computation graph lifecycle and TBPTT</b></summary>

In PyTorch, calling `loss.backward()` traverses the dynamically constructed computation graph from the loss node back to the leaves, and by default **frees intermediate buffers** to conserve memory.

If you pass $h$ into chunk $k+1$ without calling `.detach()`:
1. Chunk $k+1$ considers $h$ to be an intermediate node connected to chunk $k$'s graph.
2. When `loss.backward()` is called on chunk $k+1$, autograd attempts to differentiate through chunk $k$'s graph, whose intermediate activation buffers were already destroyed during chunk $k$'s `backward()` call.
3. PyTorch raises: `RuntimeError: Trying to backward through the graph a second time...`.

Calling `h = h.detach()` creates a new tensor that shares the underlying storage and numerical values with $h$, but has `requires_grad=False` and `grad_fn=None`, effectively severing the history while preserving the forward state.
</details>

## N5 — One LSTM Step from the Six Equations *(Long Short-Term Memory: Retaining, Writing, and Exposing State)*

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{x}_t W_{xf} + \mathbf{h}_{t-1} W_{hf} + \mathbf{b}_f) &
\mathbf{i}_t &= \sigma(\mathbf{x}_t W_{xi} + \mathbf{h}_{t-1} W_{hi} + \mathbf{b}_i)\\
\mathbf{o}_t &= \sigma(\mathbf{x}_t W_{xo} + \mathbf{h}_{t-1} W_{ho} + \mathbf{b}_o) &
\tilde{\mathbf{c}}_t &= \tanh(\mathbf{x}_t W_{xc} + \mathbf{h}_{t-1} W_{hc} + \mathbf{b}_c)\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t &
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t)
\end{aligned}
$$

### 5.1 A gate is an elementwise mask *(Slide: "Soft Gates Provide Differentiable Control")*

In [ ]:
g = torch.tensor([1.0, 0.0, 1.0]); s = torch.tensor([5.0, 2.0, 7.0])
print("g ⊙ s =", g * s)                               # slide E03: (5, 0, 7)

In [ ]:
torch.manual_seed(0)
d, hdim = 3, 4
P = {k: torch.randn(d, hdim) * 0.5 for k in ["xf", "xi", "xo", "xc"]}
P.update({k: torch.randn(hdim, hdim) * 0.5 for k in ["hf", "hi", "ho", "hc"]})
P.update({k: torch.zeros(hdim) for k in ["f", "i", "o", "c"]})

def lstm_step(x, h, c, P):
    f = torch.sigmoid(x @ P["xf"] + h @ P["hf"] + P["f"])   # retain
    i = torch.sigmoid(x @ P["xi"] + h @ P["hi"] + P["i"])   # write
    o = torch.sigmoid(x @ P["xo"] + h @ P["ho"] + P["o"])   # expose
    c_tilde = torch.tanh(x @ P["xc"] + h @ P["hc"] + P["c"])  # learned candidate
    c = f * c + i * c_tilde
    h = o * torch.tanh(c)
    return h, c, dict(f=f, i=i, o=o, c_tilde=c_tilde)

x = torch.randn(d); h0 = torch.zeros(hdim); c0 = torch.zeros(hdim)
h1, c1, gates = lstm_step(x, h0, c0, P)
for k, v in gates.items():
    print(f"{k:>7}: {v.numpy().round(3)}")
print(f"{'c_1':>7}: {c1.numpy().round(3)}\n{'h_1':>7}: {h1.numpy().round(3)}")

### 5.2 Interactive: control the forget gate
We now **hold the gates fixed** (a toy, not a trained LSTM): write a value once at step 1, then for $T$ steps set $\mathbf{i}=0$ and $\mathbf{f}=f$.
After $T$ steps the cell state is $f^{T} c_1$, and on this **direct path** $\partial c_t/\partial c_{t-1} = f$ (see slide *"Gradients Can Flow Along the Cell-State Path"*).
In a learned LSTM the gates depend on $\mathbf{h}_{t-1}$, which adds further gradient paths; this toy shows only the direct one.

In [ ]:
def carry(T=30, f=0.95):
    c1 = torch.tensor(1.0, requires_grad=True)
    c, cs = c1, [c1.item()]
    for _ in range(T):
        c = f * c + 0.0 * torch.tanh(torch.tensor(0.3))  # input gate i = 0: nothing written
        cs.append(c.item())                              # c_1 ... c_{T+1}: includes the final update
    c.backward()
    plt.figure(figsize=(6, 3.2))
    plt.plot(cs, "o-", color="goldenrod", label="cell state $c_t$")
    plt.ylim(0, 1.05); plt.xlabel("step t"); plt.legend()
    plt.title(f"f={f}:  after {T} steps c = {c.item():.3f}, dc/dc_1 = {c1.grad.item():.3f} = f^{T}")
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()

if HAVE_WIDGETS:
    interact(carry, T=IntSlider(30, 5, 100, 5), f=FloatSlider(value=0.95, min=0.5, max=1.0, step=0.01))
else:
    for f in (0.8, 0.95, 1.0):
        carry(f=f)

**Check yourself.** Compare $f=1$ with the counter of N2.1 and with the plain RNN of N4.2. Which one keeps both the value and the gradient? What does the output gate change — $\mathbf{c}_t$ or only what is exposed as $\mathbf{h}_t$?

<details>
<summary><b>Click to expand: Solution & explanation</b></summary>

- **State and gradient preservation:**
  - In the **counter (N2.1)** with $r=1$ and identity activation, values accumulate without bound and the gradient factor is $1$, but there is no nonlinearity to learn complex functions.
  - In the **plain RNN (N4.2)**, $\tanh$ provides nonlinearity, but its derivative $\le 1$ causes exponential decay of gradients through time.
  - In the **LSTM cell with $f=1$ and $i=0$**, the cell state $\mathbf{c}_t = \mathbf{c}_{t-1}$ acts as a linear conveyor belt (Hochreiter & Schmidhuber's *Constant Error Carousel*). The Jacobian is $\frac{\partial \mathbf{c}_t}{\partial \mathbf{c}_{t-1}} = I$, allowing gradients to propagate across arbitrary time spans with zero vanishing or explosion.
- **Role of the output gate $\mathbf{o}_t$:** The output gate $\mathbf{o}_t$ filters only what is exposed to the outside world as $\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$ (and passed to the prediction head or subsequent layers). It does **not** erase or modify the stored cell state $\mathbf{c}_t$! Unexposed information remains safely preserved in $\mathbf{c}_t$ and can be read out at later time steps.
</details>

## N6 — The PyTorch LSTM Interface *(Long Short-Term Memory & Fixed-Context Encoder-Decoder Models)*

`nn.LSTM` returns `output, (h_n, c_n)`:
- `output`: all exposed states $\mathbf{h}_1..\mathbf{h}_T$ (RNN*), shape `(B, T, h)` with `batch_first=True`;
- `h_n`, `c_n`: final exposed and cell state, shape `(layers × directions, B, h)` — `batch_first` does **not** change this layout.

The four gate blocks are stacked in the order **i, f, g (= candidate), o**, each transposed as in N2.

In [ ]:
lstm = nn.LSTM(input_size=d, hidden_size=hdim, batch_first=True)
for name, w in lstm.named_parameters():
    print(f"{name:14s} {tuple(w.shape)}")

# copy our manual parameters into the module
with torch.no_grad():
    lstm.weight_ih_l0.copy_(torch.cat([P[k].T for k in ["xi", "xf", "xc", "xo"]]))
    lstm.weight_hh_l0.copy_(torch.cat([P[k].T for k in ["hi", "hf", "hc", "ho"]]))
    lstm.bias_ih_l0.zero_(); lstm.bias_hh_l0.zero_()

X = torch.randn(1, 5, d)                               # (B=1, T=5, d)
output, (h_n, c_n) = lstm(X)
print("output", tuple(output.shape), "h_n", tuple(h_n.shape), "c_n", tuple(c_n.shape))

h, c = torch.zeros(hdim), torch.zeros(hdim)
for x_t in X[0]:
    h, c, _ = lstm_step(x_t, h, c, P)
assert torch.allclose(h, h_n[0, 0], atol=1e-6) and torch.allclose(c, c_n[0, 0], atol=1e-6)
assert torch.allclose(output[0, -1], h_n[0, 0])
print("nn.LSTM matches the six equations.")

`nn.LSTM` does **not** return gate activations; to inspect them, use a manual cell as in N5.

<details>
<summary><b>Click to expand: Why PyTorch packs weights into (i, f, g, o)</b></summary>

In modern deep learning libraries, running four separate matrix-vector multiplications for $\mathbf{i}_t, \mathbf{f}_t, \tilde{\mathbf{c}}_t, \mathbf{o}_t$ would launch four separate GPU/CPU kernel calls with high memory overhead.

PyTorch fuses these operations into a single large matrix multiplication by stacking the weights:
$$W_{ih} = \begin{bmatrix} W_{xi} \\ W_{xf} \\ W_{xc} \\ W_{xo} \end{bmatrix} \in \mathbb{R}^{4h \times d}, \qquad W_{hh} = \begin{bmatrix} W_{hi} \\ W_{hf} \\ W_{hc} \\ W_{ho} \end{bmatrix} \in \mathbb{R}^{4h \times h}$$
PyTorch uses the historical internal convention `(i, f, g, o)` where `g` denotes the candidate cell state $\tilde{\mathbf{c}}_t$. The single GEMM outputs a tensor of size `(B, 4*h)`, which is then sliced into four chunks of size `(B, h)` for elementwise activations.
</details>

**Check yourself.** How many parameters does a one-layer LSTM have compared with the vanilla RNN of N2, for the same $d$ and $h$? (Answer: four times as many, $4(dh+h^2+h)$, plus PyTorch's second bias.)
Replace `nn.RNN` by `nn.LSTM` in `CharLM` of N3 (remember that the returned state becomes a pair) and retrain.

<details>
<summary><b>Click to expand: Detailed parameter calculation & CharLM LSTM adapter</b></summary>

- **Parameter count comparison:**
  - Vanilla RNN: $W_{xh} (dh) + W_{hh} (h^2) + \mathbf{b} (h) = dh + h^2 + h$.
  - LSTM: Has 4 sets of transitions (input gate, forget gate, candidate state, output gate):
    $$4 \times (d \cdot h + h^2) = 4(dh + h^2)$$
    PyTorch defines two separate bias vectors per layer (`bias_ih_l0` and `bias_hh_l0`), each of size $4h$, giving $8h$ bias parameters. Total PyTorch parameters: $4dh + 4h^2 + 8h$.
- **CharLM with LSTM:** To switch `CharLM` to LSTM, update `__init__` and `forward`:
  ```python
  class CharLSTM(nn.Module):
      def __init__(self, V, d=16, hdim=64):
          super().__init__()
          self.emb = nn.Embedding(V, d)
          self.lstm = nn.LSTM(d, hdim, batch_first=True)
          self.head = nn.Linear(hdim, V)
      def forward(self, x, state=None):          # state is now a tuple (h, c)
          H, state = self.lstm(self.emb(x), state)
          return self.head(H), state
  ```
</details>

**Outlook (Movement: Fixed-Context Encoder-Decoder Models, L08).** An encoder LSTM can compress a source sentence into its final $(\mathbf{h}_T,\mathbf{c}_T)$ — a fixed-size context $\mathbf{z}$ for a decoder (as on slide *"Translating With an Encoder-Decoder"*). Attention (Lecture 08) removes this bottleneck.

## N7 — Deep Dive: Replaying Karpathy's *War and Peace* LSTM *(Using Recurrent Models: Learning to Generate Text)*

The slide *"Learning to Generate Text"* quotes samples from Andrej Karpathy's 2015 blog post *The Unreasonable Effectiveness of Recurrent Neural Networks*, from a character LSTM trained on Tolstoy's *War and Peace*:
- **After 100 iterations:** `tyntd-iafhatawiaoihrdemot lytdws e ,tfti, astai f ogoh`
- **After 2000 iterations:** `"Why do what that day," replied Natasha, and wishing to himself the fact the`

Here you replay the experiment with a character LSTM (N7.1) and the peephole variant from the appendix (N7.2).
The code uses a GPU (`cuda` or Apple `mps`) if available; on Colab choose *Runtime → Change runtime type → GPU*.

### N7.0 — Data: clean the line ends and hard wraps
Project Gutenberg texts use Windows line ends (`\r\n`) and break lines after about 70 characters.
We therefore turn `\r\n` into `\n` first (otherwise the character before each `\n` is `\r`, and a pattern looking for "text, newline, text" never matches),
then replace every *single* newline by a space and keep blank lines as paragraph breaks.
The last 10% of the book is held out for validation; perplexity is measured there.

In [ ]:
import urllib.request, re, os, time

# 1. Download War and Peace (Project Gutenberg #2600) and strip the license header/footer
wp_url = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
wp_file = "warpeace_input.txt"
if not os.path.exists(wp_file):
    print("Downloading War and Peace from Project Gutenberg...")
    req = urllib.request.Request(wp_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        raw = resp.read().decode("utf-8")
    s = re.search(r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK .* \*\*\*", raw)
    e = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK .* \*\*\*", raw)
    raw = raw[s.end():e.start()] if (s and e) else raw
    with open(wp_file, "w", encoding="utf-8", newline="") as f:   # keep the file byte-identical
        f.write(raw)
with open(wp_file, encoding="utf-8", newline="") as f:             # newline="": see the \r\n as they are
    raw = f.read()

# 2. Clean: Gutenberg uses Windows line ends (\r\n) and hard-wraps lines at ~70 characters.
text = raw.replace("\r\n", "\n").replace("\r", "\n")              # order matters: \r first, else \n never follows \S
text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)                       # single newline = hard wrap -> space
text = re.sub(r"\n{3,}", "\n\n", text)                             # blank line = paragraph break
text = re.sub(r" {2,}", " ", text).strip()
print(f"raw: {len(raw):,} chars, {raw.count(chr(10)):,} newlines | clean: {len(text):,} chars, {text.count(chr(10)):,} newlines")
print("raw:  ", repr(raw[100000:100160])); print("clean:", repr(text[98000:98160]))

# 3. Vocabulary, train/validation split (last 10% held out), device
chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"V = {V} characters | train {len(train_data):,} | val {len(val_data):,} | device: {device}")

def get_batch(split, B=64, T=100):
    d = train_data if split == "train" else val_data
    starts = torch.randint(0, len(d) - T - 1, (B,))
    x = torch.stack([d[s:s + T] for s in starts])
    y = torch.stack([d[s + 1:s + T + 1] for s in starts])          # shifted by one, as in N1
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss(model, batches=20):
    model.eval()
    g = torch.Generator().manual_seed(0)                            # same validation windows every time
    losses = []
    for _ in range(batches):
        starts = torch.randint(0, len(val_data) - 101, (64,), generator=g)
        x = torch.stack([val_data[s:s + 100] for s in starts]).to(device)
        y = torch.stack([val_data[s + 1:s + 101] for s in starts]).to(device)
        scores, _ = model(x)
        losses.append(F.cross_entropy(scores.reshape(-1, V), y.reshape(-1)).item())
    model.train()
    return sum(losses) / len(losses)

@torch.no_grad()
def sample(model, prompt="“Why ", length=200, temp=0.7):
    model.eval()
    x = torch.tensor([[stoi[c] for c in prompt]], device=device)
    scores, state = model(x)                                        # read the prompt into the state
    out = list(prompt)
    for _ in range(length):
        probs = F.softmax(scores[0, -1] / temp, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        out.append(itos[nxt])
        scores, state = model(torch.tensor([[nxt]], device=device), state)
    model.train()
    return "".join(out)

def train(model, steps, every, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    t0 = time.time()
    print(f"step    0 | val loss {val_loss(model):.3f} | val PPL {math.exp(val_loss(model)):6.2f}")
    for step in range(1, steps + 1):
        x, y = get_batch("train")
        scores, _ = model(x)
        loss = F.cross_entropy(scores.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)          # N4: clip exploding gradients
        opt.step()
        if step % every == 0:
            vl = val_loss(model)
            print(f"\nstep {step:4d} | val loss {vl:.3f} | val PPL {math.exp(vl):6.2f} | {time.time() - t0:5.0f} s")
            print(sample(model))
    return model

### N7.1 — A character LSTM with `nn.LSTM`
The model is the `CharLM` of N3 with `nn.LSTM` (one layer, 1024 hidden units) in place of `nn.RNN`.
Each report shows the **validation** perplexity and a sample at temperature 0.7, started from the prompt `“Why `.

In [ ]:
class CharLSTM(nn.Module):
    """Embedding -> nn.LSTM -> linear scores over the V characters."""
    def __init__(self, V, emb_dim=64, hidden_dim=1024, num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(V, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden_dim, V)
    def forward(self, x, state=None):
        H, state = self.lstm(self.emb(x), state)
        return self.head(H), state

STEPS = 2000          # a few minutes on a GPU; reduce to 300 on a CPU
torch.manual_seed(42)
model = CharLSTM(V).to(device)
print("untrained:", sample(model, length=80))
train(model, STEPS, every=STEPS // 5);

### N7.2 — Variant: peephole connections *(Appendix: "Peephole Connections Let Gates Read the Cell")*
In the standard LSTM (N5), the gates read only $\mathbf{x}_t$ and $\mathbf{h}_{t-1}$. With **peepholes** they also read the cell state, through a diagonal (elementwise) weight vector $\mathbf{p}$:

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{x}_t W_{xf} + \mathbf{h}_{t-1} W_{hf} + \mathbf{c}_{t-1}\odot\mathbf{p}_f + \mathbf{b}_f) &
\mathbf{i}_t &= \sigma(\mathbf{x}_t W_{xi} + \mathbf{h}_{t-1} W_{hi} + \mathbf{c}_{t-1}\odot\mathbf{p}_i + \mathbf{b}_i)\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t &
\mathbf{o}_t &= \sigma(\mathbf{x}_t W_{xo} + \mathbf{h}_{t-1} W_{ho} + \mathbf{c}_{t}\odot\mathbf{p}_o + \mathbf{b}_o)
\end{aligned}
$$

$\tilde{\mathbf{c}}_t$ and $\mathbf{h}_t$ are unchanged. Note the order: the output gate peeks at the *new* $\mathbf{c}_t$.
`nn.LSTM` has no peephole option, so we write the cell ourselves and unroll it with a Python loop over time, exactly the unrolled graph of the lecture.
With $\mathbf{p}=\mathbf{0}$ (our initialization) the cell *is* the standard LSTM. Same size, same steps, same seed as N7.1.

In [ ]:
class PeepholeLSTMCell(nn.Module):
    """One LSTM step whose gates also read the cell state (diagonal peephole weights)."""
    def __init__(self, d, hdim):
        super().__init__()
        self.Wx = nn.Linear(d, 4 * hdim)                      # x_t [W_xf W_xi W_xo W_xc] + biases
        self.Wh = nn.Linear(hdim, 4 * hdim, bias=False)       # h_{t-1} [W_hf W_hi W_ho W_hc]
        self.p = nn.Parameter(torch.zeros(3, hdim))           # peepholes p_f, p_i, p_o (start at 0 = standard LSTM)
    def forward(self, x, h, c):
        zf, zi, zo, zc = (self.Wx(x) + self.Wh(h)).chunk(4, dim=-1)
        f = torch.sigmoid(zf + c * self.p[0])                 # retain: reads c_{t-1}
        i = torch.sigmoid(zi + c * self.p[1])                 # write:  reads c_{t-1}
        c = f * c + i * torch.tanh(zc)
        o = torch.sigmoid(zo + c * self.p[2])                 # expose: reads the new c_t
        return o * torch.tanh(c), c

class PeepholeCharLSTM(nn.Module):
    def __init__(self, V, emb_dim=64, hidden_dim=1024, num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(V, emb_dim)
        self.cells = nn.ModuleList([PeepholeLSTMCell(emb_dim if l == 0 else hidden_dim, hidden_dim)
                                    for l in range(num_layers)])
        self.head = nn.Linear(hidden_dim, V)
    def forward(self, x, state=None):
        B, T = x.shape
        if state is None:
            z = torch.zeros(B, self.head.in_features, device=x.device)
            state = [(z, z) for _ in self.cells]
        E, outs = self.emb(x), []
        for t in range(T):                                    # explicit unrolling: a Python loop over time
            inp = E[:, t]
            new_state = []
            for cell, (h, c) in zip(self.cells, state):
                h, c = cell(inp, h, c)
                new_state.append((h, c))
                inp = h                                       # layer l's h_t is layer l+1's input
            state = new_state
            outs.append(inp)
        return self.head(torch.stack(outs, dim=1)), state

torch.manual_seed(42)
peep = PeepholeCharLSTM(V).to(device)
train(peep, STEPS, every=STEPS // 5)
print("\nlearned peephole weights (mean |p|, layer 1): "
      + ", ".join(f"{g}={peep.cells[0].p[k].abs().mean():.3f}" for k, g in enumerate("fio")))

**Look at your run.**
1. What is the validation perplexity at step 0, and how does it relate to the vocabulary size $V$ (slide *"Comparing Next-Token Predictions"*)?
2. Which units appear first in the samples (spaces, short words, punctuation, longer words, names)? At which step?
3. How many effective choices per character (slide *"How Difficult Is the Next Character?"*) does the final perplexity correspond to?
4. Do the samples hold together beyond a few words? Compare them with Karpathy's sample after 2000 iterations.
5. How do the peephole variant's perplexity, learned peephole weights, and run time compare with `nn.LSTM`? Why does the hand-written cell run at a different speed? Would one run be enough to call either variant better (appendix: Greff et al.)?

**Try it.** (a) Skip the cleaning (`text = raw`): where do newlines appear in the samples? (b) Change the temperature in `sample` to 0.3 and 1.2. (c) Set `STEPS = 3000`.